# P-Wave Moveout and Ps Conversion — GeoNet / 2011 Tohoku M9

Loads `tohoku_nz_array.npz` (run `download_data.py` first), picks P and Ps arrivals, fits linear moveout curves, and plots a three-component record section.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

: 

## 1. Load data

In [ ]:
d = np.load('tohoku_nz_array.npz', allow_pickle=True)

Z         = d['Z']            # (N stations, T samples) — vertical
R         = d['R']            # radial (toward earthquake)
T         = d['T']            # transverse
distances = d['distances']    # epicentral distance, km
times     = d['times_rel']    # seconds relative to predicted P (0 = P)
stations  = d['stations']
sr        = float(d['sampling_rate'])  # samples/sec

print(f"{Z.shape[0]} stations, {Z.shape[1]} samples each")
print(f"Distance range: {distances.min():.0f} – {distances.max():.0f} km")
print(f"Time window:    {times[0]:.0f} to {times[-1]:.0f} s relative to P")

## 2. Sort stations by distance (north → south across NZ)

In [ ]:
order     = np.argsort(distances)
Z         = Z[order]
R         = R[order]
T         = T[order]
distances = distances[order]
stations  = stations[order]

print("Stations (closest first):")
for sta, dist in zip(stations, distances):
    print(f"  {sta}  {dist:.0f} km")

## 3. Quick look — raw traces

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)

mid = len(stations) // 2   # pick a station in the middle of the array

for ax, comp, label, col in zip(axes,
                                 [Z[mid], R[mid], T[mid]],
                                 ['Z (vertical)', 'R (radial)', 'T (transverse)'],
                                 ['#2c3e50', '#c0392b', '#16a085']):
    ax.plot(times, comp, color=col, linewidth=0.8)
    ax.set_ylabel(label, fontsize=9)
    ax.axvline(0, color='gray', linewidth=0.8, linestyle=':')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time relative to predicted P (s)')
fig.suptitle(f'Station {stations[mid]}  —  {distances[mid]:.0f} km from Tohoku', fontsize=11)
plt.tight_layout()

## 4. Pick P on vertical (Z) and Ps on radial (R)

In [ ]:
from scipy.fft import next_fast_len

def time_to_idx(t_sec):
    """Convert a time in seconds (relative to P) to a sample index."""
    return int(round((t_sec - times[0]) * sr))

def receiver_function(z, r, sr, water_level=0.01, gauss_a=2.5):
    """
    Frequency-domain receiver function: deconvolve Z from R.

    RF(f) = [R(f) · Z*(f)] / [|Z(f)|² + wl]  ×  G(f)

    water_level : fraction of max |Z(f)|² used as noise floor
    gauss_a     : Gaussian width in Hz (low-pass taper; 2.5 Hz ≈ 1 s resolution)
    """
    n    = len(z)
    nfft = next_fast_len(n)
    Z_f  = np.fft.rfft(z, n=nfft)
    R_f  = np.fft.rfft(r, n=nfft)
    ZZ   = (Z_f * np.conj(Z_f)).real          # |Z(f)|²
    wl   = water_level * ZZ.max()              # water-level floor
    RF_f = (R_f * np.conj(Z_f)) / (ZZ + wl)   # spectral division
    # Gaussian low-pass: suppresses high-freq noise amplified by division
    freqs = np.fft.rfftfreq(nfft, d=1.0 / sr)
    RF_f *= np.exp(-(freqs ** 2) / (4.0 * gauss_a ** 2))
    return np.fft.irfft(RF_f, n=nfft)[:n]

N = len(stations)
p_picks  = np.empty(N)
ps_picks = np.empty(N)
RF       = np.empty_like(Z)   # receiver functions, one per station

P_WIN  = (0.0, 8.0)   # search window for P peak on Z
PS_WIN = (2.0, 5.0)   # search window for Ps on RF after P (NZ Moho → 3–4 s)

for i in range(N):
    # P: max amplitude on Z (unchanged)
    i0, i1    = time_to_idx(P_WIN[0]), time_to_idx(P_WIN[1])
    p_picks[i] = times[i0 + np.argmax(np.abs(Z[i, i0:i1]))]

    # Receiver function: deconvolve Z from R for the full trace
    RF[i] = receiver_function(Z[i], R[i], sr)

    # Ps: positive peak in receiver function within PS_WIN of P pick
    # (Ps shows as a positive pulse in RF for standard crustal geometry)
    j0 = time_to_idx(p_picks[i] + PS_WIN[0])
    j1 = time_to_idx(p_picks[i] + PS_WIN[1])
    j0, j1 = max(0, j0), min(Z.shape[1] - 1, j1)
    ps_picks[i] = times[j0 + np.argmax(RF[i, j0:j1])]

tau_ps = ps_picks - p_picks
print(f"P picks  range: {p_picks.min():.1f} – {p_picks.max():.1f} s")
print(f"Ps picks range: {ps_picks.min():.1f} – {ps_picks.max():.1f} s")
print(f"τ_Ps     range: {tau_ps.min():.1f} – {tau_ps.max():.1f} s  (median {np.median(tau_ps):.1f} s)")

# ── quick QC: overlay RF on raw R for the middle station ─────────────────────
mid = len(stations) // 2
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
mask_qc = (times >= -2) & (times <= 12)
for ax, tr, label, col in [
        (ax1, R[mid],  'Raw R',                     '#c0392b'),
        (ax2, RF[mid], 'Receiver function (R/Z)',   '#8e44ad')]:
    ax.plot(times[mask_qc], tr[mask_qc], color=col, linewidth=0.9)
    ax.axvline(p_picks[mid],  color='steelblue', linewidth=1.2, linestyle='--', label='P pick')
    ax.axvline(ps_picks[mid], color='gold',      linewidth=1.2, linestyle=':',  label='Ps pick (RF)')
    ax.axhline(0, color='gray', linewidth=0.4)
    ax.set_ylabel(label, fontsize=9)
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)
ax2.set_xlabel('Time relative to predicted P (s)')
fig.suptitle(f'Station {stations[mid]}  —  Raw R vs Receiver Function', fontsize=11)
plt.tight_layout()


## 5. Fit linear moveout to P and Ps

In [ ]:
def linear(dist, a, v):
    """t = a + d/v"""
    return a + dist / v

popt_p,  pcov_p  = curve_fit(linear, distances, p_picks,  p0=[0.0, 8.0])
popt_ps, pcov_ps = curve_fit(linear, distances, ps_picks, p0=[5.0, 8.0])

v_p  = popt_p[1];   v_p_err  = np.sqrt(pcov_p[1, 1])
v_ps = popt_ps[1];  v_ps_err = np.sqrt(pcov_ps[1, 1])

d_mid  = np.median(distances)
d_line = np.linspace(distances.min(), distances.max(), 200)

# τ_Ps: use per-station median delay to avoid ~7700 km lever-arm amplification
# (fitting two nearly-parallel lines and evaluating at d_mid magnifies tiny
#  slope differences into hundreds-of-seconds errors)
tau_fit = float(np.median(tau_ps))

# Rough crustal thickness from τ_Ps
Vp, Vs, p = 6.4, 3.7, 0.045   # km/s, km/s, s/km  (p ≈ 0.045 at ~75°)
eta_S = np.sqrt(1/Vs**2 - p**2)
eta_P = np.sqrt(1/Vp**2 - p**2)
H_est = tau_fit / (eta_S - eta_P)

print(f"P  apparent velocity : {v_p:.2f} ± {v_p_err:.2f} km/s")
print(f"Ps apparent velocity : {v_ps:.2f} ± {v_ps_err:.2f} km/s")
print(f"τ_Ps (median)        : {tau_fit:.2f} s")
print(f"Crustal thickness H  : {H_est:.0f} km")


## 6. Three-component record section

In [ ]:
SCALE   = 60.0    # km — visual half-amplitude
d_line  = np.linspace(distances.min(), distances.max(), 200)

fig, axes = plt.subplots(1, 3, figsize=(15, 10), sharey=True)

comp_data  = [Z,         R,              T]
comp_names = ['Z (vertical)', 'R (radial)', 'T (transverse)']
comp_cols  = ['#2c3e50',   '#c0392b',   '#16a085']

for ax, comp, name, col in zip(axes, comp_data, comp_names, comp_cols):
    for i in range(N):
        ax.plot(times, comp[i] * SCALE + distances[i],
                color=col, linewidth=0.6, alpha=0.75)

    # P picks
    ax.scatter(p_picks, distances, marker='|', s=80,
               linewidths=1.5, color='steelblue', zorder=4, label='P')

    # Ps picks on radial only
    if 'radial' in name:
        ax.scatter(ps_picks, distances, marker='|', s=80,
                   linewidths=1.5, color='gold', zorder=4, label='Ps')

    # Moveout lines
    ax.plot(linear(d_line, *popt_p), d_line,
            color='steelblue', linewidth=1.8, linestyle='--',
            label=f'P  v={v_p:.1f} km/s')
    if 'radial' in name:
        ax.plot(linear(d_line, *popt_ps), d_line,
                color='gold', linewidth=1.8, linestyle='--',
                label=f'Ps τ={tau_fit:.1f} s')

    ax.axvline(0, color='gray', linewidth=0.8, linestyle=':')
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Time relative to predicted P (s)', fontsize=10)
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8, loc='upper left')
    ax.set_xlim(times[0], times[-1])

axes[0].set_ylabel('Epicentral distance (km)', fontsize=11)
axes[0].invert_yaxis()

fig.suptitle('2011 Tohoku M9.0 — GeoNet NZ array  (0.5–2 Hz)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('record_section.png', dpi=150, bbox_inches='tight')

## 7. Moveout fit: P vs Ps

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 9),
                                gridspec_kw={'height_ratios': [3, 1]})

# Top: picks and fits
ax1.scatter(distances, p_picks,  color='steelblue', s=55, zorder=3,
            edgecolors='white', linewidths=0.5, label='P picks  (Z)')
ax1.scatter(distances, ps_picks, color='#e67e22',   s=55, zorder=3,
            edgecolors='white', linewidths=0.5, label='Ps picks (R)')
ax1.plot(d_line, linear(d_line, *popt_p),
         color='steelblue', linewidth=2,
         label=f'P fit  v = {v_p:.2f} km/s')
ax1.plot(d_line, linear(d_line, *popt_ps),
         color='#e67e22', linewidth=2,
         label=f'Ps fit  τ = {tau_fit:.1f} s  →  H ≈ {H_est:.0f} km')

ax1.set_ylabel('Arrival time relative to predicted P (s)', fontsize=11)
ax1.set_title(f'P and Ps Moveout — τ_Ps = {tau_fit:.1f} s  →  H ≈ {H_est:.0f} km', fontsize=12)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Bottom: per-station τ_Ps (should be roughly flat)
ax2.scatter(distances, tau_ps, color='#e67e22', s=40,
            edgecolors='white', linewidths=0.4)
ax2.axhline(np.median(tau_ps), color='gray', linewidth=1.5, linestyle='--',
            label=f'Median = {np.median(tau_ps):.1f} s')
ax2.set_xlabel('Epicentral distance (km)', fontsize=11)
ax2.set_ylabel('τ_Ps  (s)', fontsize=11)
ax2.set_title('P-to-Ps delay per station', fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('moveout_fit.png', dpi=150, bbox_inches='tight')

## 8. Summary

In [ ]:
print("Results")
print("-------")
print(f"Stations             : {N}")
print(f"P apparent velocity  : {v_p:.2f} ± {v_p_err:.2f} km/s")
print(f"Ps apparent velocity : {v_ps:.2f} ± {v_ps_err:.2f} km/s")
print(f"τ_Ps                 : {tau_fit:.2f} s")
print(f"Crustal thickness H  : {H_est:.0f} km  (assuming Vp={Vp}, Vs={Vs} km/s)")

In [ ]:

"""
3-Component PCA orientation analysis
-------------------------------------
For each station, run PCA on [Z, N, E] in a one-sided window
  [p_pick, p_pick + WIN_LEN]
using only the FORWARD half of the P wavetrain to avoid the negative
bounce cycle that often dominates a symmetric ±1.5 s window.

Ambiguity resolution:
  - Check the sign of Z at the *first sample* of the window (true first motion)
  - Force PC1[Z] to match that sign
  (argmax|Z| can land on a negative cycle → symmetric window would flip polarity)

Reliability filter:
  - PC1 variance fraction > 60 %
  - Incidence angle 5–40°
"""
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

d2      = np.load('tohoku_nz_array.npz', allow_pickle=True)
baz_cat = d2['baz'][np.argsort(d2['distances'])]

def rt_to_ne(r, t, baz_deg):
    """ObsPy inverse: RT → NE."""
    b = np.radians(baz_deg)
    return -r*np.cos(b) + t*np.sin(b), -r*np.sin(b) - t*np.cos(b)

WIN_LEN = 2.0   # seconds AFTER the P pick (one-sided forward window)

nsta      = len(stations)
var_frac  = np.empty((nsta, 3))
az_pca    = np.empty(nsta)
inc_pca   = np.empty(nsta)

for i in range(nsta):
    n_comp, e_comp = rt_to_ne(R[i], T[i], baz_cat[i])

    # One-sided window: from P onset forward
    i0 = max(0, int(round((p_picks[i] - times[0]) * sr)))
    i1 = min(len(times), int(round((p_picks[i] + WIN_LEN - times[0]) * sr)))

    data = np.column_stack([Z[i, i0:i1], n_comp[i0:i1], e_comp[i0:i1]])
    data -= data.mean(axis=0)
    cov  = data.T @ data

    eigvals, eigvecs = np.linalg.eigh(cov)
    idx = np.argsort(eigvals)[::-1]
    eigvals = eigvals[idx]; eigvecs = eigvecs[:, idx]
    var_frac[i] = eigvals / eigvals.sum()

    pc1 = eigvecs[:, 0]

    # Use true first-motion polarity: sign of Z at the very first sample
    # (not argmax which may be on a negative cycle)
    first_motion_sign = np.sign(Z[i, i0])
    if first_motion_sign == 0:
        first_motion_sign = 1.0
    if np.sign(pc1[0]) != first_motion_sign:
        pc1 = -pc1

    az_pca[i]  = np.degrees(np.arctan2(pc1[2], pc1[1])) % 360
    horiz_amp  = np.sqrt(pc1[1]**2 + pc1[2]**2)
    inc_pca[i] = np.degrees(np.arctan2(horiz_amp, abs(pc1[0])))

def wrap(d):
    return ((d + 180) % 360) - 180

delta_baz = wrap(az_pca - baz_cat)

# ── reliability mask ──────────────────────────────────────────────────────────
PC1_VAR_MIN = 60.0
INC_MIN, INC_MAX = 5.0, 40.0

reliable = (
    (var_frac[:, 0] * 100 >= PC1_VAR_MIN) &
    (inc_pca >= INC_MIN) &
    (inc_pca <= INC_MAX)
)

# ── print table ───────────────────────────────────────────────────────────────
print(f"{'Station':<12} {'Cat baz':>8} {'PCA baz':>8} {'Δbaz':>7} "
      f"{'Inc°':>6}  {'PC1%':>6}  {'Quality'}")
print("-" * 65)
for i, sta in enumerate(stations):
    qual = "OK" if reliable[i] else "low PC1" if var_frac[i,0]*100 < PC1_VAR_MIN else "bad inc"
    print(f"{sta:<12} {baz_cat[i]:>8.1f}° {az_pca[i]:>7.1f}° "
          f"{delta_baz[i]:>+7.1f}° {inc_pca[i]:>5.1f}°  "
          f"{var_frac[i,0]*100:>5.1f}%  {qual}")

n_rel = reliable.sum()
print(f"\n{n_rel}/{nsta} stations pass reliability filter "
      f"(PC1 ≥ {PC1_VAR_MIN}%, inc {INC_MIN}–{INC_MAX}°)")
if n_rel > 0:
    print(f"Reliable stations — mean Δbaz : {np.mean(delta_baz[reliable]):+.1f}°")
    print(f"Reliable stations — std  Δbaz : {np.std(delta_baz[reliable]):.1f}°")
    print(f"Reliable stations — median    : {np.median(delta_baz[reliable]):+.1f}°")

# ── figure ────────────────────────────────────────────────────────────────────
sta_labels = [s.split('.')[1] for s in stations]
x = np.arange(nsta)

fig = plt.figure(figsize=(13, 10))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.5, wspace=0.35)

ax_var = fig.add_subplot(gs[0, :])
ax_var.bar(x, var_frac[:,0]*100, label='PC1', color='#2c3e50')
ax_var.bar(x, var_frac[:,1]*100, bottom=var_frac[:,0]*100, label='PC2', color='#3498db')
ax_var.bar(x, var_frac[:,2]*100, bottom=(var_frac[:,0]+var_frac[:,1])*100,
           label='PC3', color='#85c1e9')
ax_var.axhline(PC1_VAR_MIN, color='#e74c3c', linewidth=1.2, linestyle='--',
               label=f'Reliability threshold ({PC1_VAR_MIN}%)')
ax_var.set_xticks(x); ax_var.set_xticklabels(sta_labels, rotation=45, ha='right', fontsize=7)
ax_var.set_ylabel('Variance (%)'); ax_var.set_title('Variance explained by each PC')
ax_var.legend(fontsize=8, loc='lower right'); ax_var.grid(True, alpha=0.3, axis='y')

ax_dbaz = fig.add_subplot(gs[1, :])
colors = []
for i in range(nsta):
    if not reliable[i]:
        colors.append('#cccccc')
    elif abs(delta_baz[i]) > 20:
        colors.append('#e74c3c')
    else:
        colors.append('#27ae60')
ax_dbaz.bar(x, delta_baz, color=colors)
ax_dbaz.axhline(0,    color='k',    linewidth=0.8)
ax_dbaz.axhline( 20,  color='gray', linewidth=0.8, linestyle='--')
ax_dbaz.axhline(-20,  color='gray', linewidth=0.8, linestyle='--', label='±20°')
from matplotlib.patches import Patch
ax_dbaz.legend(handles=[
    Patch(color='#27ae60', label='Reliable, |Δ| ≤ 20°'),
    Patch(color='#e74c3c', label='Reliable, |Δ| > 20°'),
    Patch(color='#cccccc', label='Unreliable'),
], fontsize=8, loc='lower right')
ax_dbaz.set_xticks(x); ax_dbaz.set_xticklabels(sta_labels, rotation=45, ha='right', fontsize=7)
ax_dbaz.set_ylabel('PCA baz − catalog baz  (°)')
ax_dbaz.set_title('Back-azimuth misfit  (one-sided window, first-motion polarity)')
ax_dbaz.set_ylim(-185, 185); ax_dbaz.grid(True, alpha=0.3, axis='y')

ax_inc = fig.add_subplot(gs[2, 0])
inc_colors = ['#8e44ad' if reliable[i] else '#cccccc' for i in range(nsta)]
ax_inc.bar(x, inc_pca, color=inc_colors)
ax_inc.axhline(INC_MIN, color='#e74c3c', linewidth=1, linestyle='--')
ax_inc.axhline(INC_MAX, color='#e74c3c', linewidth=1, linestyle='--',
               label=f'Valid range {INC_MIN}–{INC_MAX}°')
ax_inc.set_xticks(x); ax_inc.set_xticklabels(sta_labels, rotation=45, ha='right', fontsize=7)
ax_inc.set_ylabel('Incidence angle (°)'); ax_inc.set_title('PC1 incidence angle')
ax_inc.legend(fontsize=8); ax_inc.grid(True, alpha=0.3, axis='y')

ax_sc = fig.add_subplot(gs[2, 1])
ax_sc.scatter(var_frac[~reliable, 0]*100, np.abs(delta_baz[~reliable]),
              s=40, color='#cccccc', edgecolors='k', linewidths=0.3,
              zorder=2, label='Unreliable')
sc = ax_sc.scatter(var_frac[reliable, 0]*100, np.abs(delta_baz[reliable]),
                   c=inc_pca[reliable], cmap='plasma', s=70,
                   edgecolors='k', linewidths=0.4, zorder=3, label='Reliable')
for i in np.where(reliable)[0]:
    ax_sc.annotate(stations[i].split('.')[1],
                   (var_frac[i,0]*100, abs(delta_baz[i])),
                   fontsize=6, xytext=(3,3), textcoords='offset points')
plt.colorbar(sc, ax=ax_sc, label='Inc angle (°)')
ax_sc.axvline(PC1_VAR_MIN, color='#e74c3c', linewidth=0.8, linestyle='--')
ax_sc.axhline(20, color='gray', linewidth=0.8, linestyle='--')
ax_sc.set_xlabel('PC1 variance (%)'); ax_sc.set_ylabel('|Δbaz| (°)')
ax_sc.set_title('PC1 quality vs Δbaz')
ax_sc.legend(fontsize=8); ax_sc.grid(True, alpha=0.3)

fig.suptitle('3-Component PCA — one-sided window + first-motion polarity',
             fontsize=11, y=1.01)
plt.savefig('pca_orientation.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
"""
PCA-corrected rotation
-----------------------
Use the PCA-estimated back-azimuth (az_pca) to re-rotate R/T into the
true radial/transverse frame, then compare to the catalog-baz rotation.

Strategy:
  1. Back-rotate current R, T → sensor-frame N', E'  (using baz_cat, same as
     download_data.py did in reverse)
  2. Re-rotate N', E' → R_pca, T_pca  using az_pca
  3. Compare the two radial channels around the P and Ps picks
"""
import numpy as np
import matplotlib.pyplot as plt
from obspy.signal.rotate import rotate_ne_rt

# ── back-rotate to sensor frame then re-rotate with PCA azimuth ──────────────
def rt_to_ne(r, t, baz_deg):
    """ObsPy inverse: RT → NE."""
    b = np.radians(baz_deg)
    n = -r * np.cos(b) + t * np.sin(b)
    e = -r * np.sin(b) - t * np.cos(b)
    return n, e

nsta  = len(stations)
R_pca = np.empty_like(R)
T_pca = np.empty_like(T)

for i in range(nsta):
    n_sens, e_sens = rt_to_ne(R[i], T[i], baz_cat[i])
    r_c, t_c = rotate_ne_rt(n_sens, e_sens, az_pca[i])
    R_pca[i] = r_c
    T_pca[i] = t_c

# ── polar comparison: az_pca vs baz_cat ──────────────────────────────────────
fig_pol, ax_pol = plt.subplots(subplot_kw=dict(projection='polar'), figsize=(6, 6))
angles_rad_cat = np.radians(baz_cat)
angles_rad_pca = np.radians(az_pca)

ax_pol.scatter(angles_rad_cat, np.ones(nsta), s=60, color='#3498db',
               label='Catalog baz', zorder=3)
ax_pol.scatter(angles_rad_pca, np.ones(nsta), s=60, marker='x', color='#e74c3c',
               linewidths=1.5, label='PCA baz', zorder=4)
for i in range(nsta):
    ax_pol.plot([angles_rad_cat[i], angles_rad_pca[i]], [1, 1],
                color='gray', linewidth=0.7, alpha=0.6)

ax_pol.set_theta_zero_location('N')
ax_pol.set_theta_direction(-1)
ax_pol.set_rticks([])
ax_pol.set_title('Catalog baz (●) vs PCA baz (×)\nLines connect same station', pad=15)
ax_pol.legend(loc='lower right', bbox_to_anchor=(1.3, -0.05))
plt.tight_layout()
plt.savefig('pca_baz_polar.png', dpi=150, bbox_inches='tight')
plt.show()

# ── per-station trace comparison: catalog-R vs PCA-R in P/Ps window ──────────
# Show 6 representative stations (evenly spaced)
idx_show = np.linspace(0, nsta - 1, min(6, nsta), dtype=int)
win_start, win_end = -2.0, 12.0   # seconds around predicted P

fig2, axes2 = plt.subplots(len(idx_show), 2, figsize=(12, 2.5 * len(idx_show)),
                            sharex=True)
fig2.suptitle('Radial (R) component: catalog baz vs PCA baz\n'
              'Blue = catalog baz  |  Red = PCA baz  |  ▲ P pick  |  ● Ps pick',
              fontsize=11)

mask = (times >= win_start) & (times <= win_end)
t_win = times[mask]

for row, i in enumerate(idx_show):
    for col, (rad, label, color) in enumerate([
            (R[i],   f'Catalog baz={baz_cat[i]:.1f}°', '#3498db'),
            (R_pca[i], f'PCA baz={az_pca[i]:.1f}°  Δ={delta_baz[i]:+.1f}°', '#e74c3c')]):
        ax = axes2[row, col]
        ax.plot(t_win, rad[mask], color=color, linewidth=0.9)
        ax.axvline(p_picks[i],  color='#2c3e50', linewidth=1.2,
                   linestyle='--', label='P')
        ax.axvline(ps_picks[i], color='#e67e22', linewidth=1.2,
                   linestyle=':', label='Ps')
        ax.axhline(0, color='gray', linewidth=0.4)
        ax.set_title(f'{stations[i].split(".")[1]}  —  {label}', fontsize=8)
        ax.grid(True, alpha=0.25)
        if col == 0:
            ax.set_ylabel('Norm. amp.', fontsize=8)

axes2[-1, 0].set_xlabel('Time relative to predicted P (s)')
axes2[-1, 1].set_xlabel('Time relative to predicted P (s)')
plt.tight_layout()
plt.savefig('pca_rotation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ── summary ───────────────────────────────────────────────────────────────────
print(f"{'Station':<12} {'Cat baz':>8} {'PCA baz':>8} {'Δbaz':>7}  {'Correction applied'}")
print("-" * 55)
for i, sta in enumerate(stations):
    quality = "large" if abs(delta_baz[i]) > 20 else "small"
    print(f"{sta:<12} {baz_cat[i]:>8.1f}° {az_pca[i]:>7.1f}° {delta_baz[i]:>+7.1f}°  ({quality} correction)")


In [ ]:
"""
Systematicity of PCA baz offset  (Δbaz = az_pca − baz_cat)
------------------------------------------------------------
If the offset is driven by crustal heterogeneity or anisotropy it should
show a spatially coherent pattern (smooth gradient with latitude/distance).
If it is sensor misorientation it will be station-specific (spatially random).

Tests:
  1. t-test: is the mean Δbaz significantly non-zero?  (systematic bias)
  2. Linear regression of Δbaz vs epicentral distance  (path-length heterogeneity)
  3. Linear regression of Δbaz vs reconstructed station latitude  (spatial gradient)
  4. Map coloured by Δbaz
"""
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy import stats

# ── reconstruct station lat/lon from event + baz + distance ──────────────────
EVT_LAT, EVT_LON = 38.297, 142.373
d2      = np.load('tohoku_nz_array.npz', allow_pickle=True)
dist_deg = d2['dist_deg'][np.argsort(d2['distances'])]   # degrees, same sort

# Forward azimuth: event → station = (baz_cat + 180) % 360
faz = np.radians((baz_cat + 180) % 360)
d_rad = np.radians(dist_deg)
phi1  = np.radians(EVT_LAT)
lam1  = np.radians(EVT_LON)

sta_lat = np.degrees(np.arcsin(
    np.sin(phi1)*np.cos(d_rad) + np.cos(phi1)*np.sin(d_rad)*np.cos(faz)))
sta_lon = np.degrees(lam1 + np.arctan2(
    np.sin(faz)*np.sin(d_rad)*np.cos(phi1),
    np.cos(d_rad) - np.sin(phi1)*np.sin(np.radians(sta_lat))))

# ── statistical tests ─────────────────────────────────────────────────────────
# 1. One-sample t-test: H0 = mean(Δbaz) = 0
t_stat, p_val = stats.ttest_1samp(delta_baz, 0)

# 2. Correlation with epicentral distance
r_dist, p_dist = stats.pearsonr(distances, delta_baz)

# 3. Correlation with station latitude
r_lat, p_lat = stats.pearsonr(sta_lat, delta_baz)

# 4. Linear regression vs distance and latitude
slope_dist, int_dist, *_ = stats.linregress(distances, delta_baz)
slope_lat,  int_lat,  *_ = stats.linregress(sta_lat,   delta_baz)

print("═" * 60)
print("SYSTEMATICITY ANALYSIS OF Δbaz = PCA baz − catalog baz")
print("═" * 60)
print(f"\nMean Δbaz  : {np.mean(delta_baz):+.1f}°   (std = {np.std(delta_baz):.1f}°)")
print(f"Median Δbaz: {np.median(delta_baz):+.1f}°")
print()
print("One-sample t-test (H₀: mean Δbaz = 0)")
print(f"  t = {t_stat:.2f},  p = {p_val:.4f}  → "
      f"{'SIGNIFICANT systematic bias' if p_val < 0.05 else 'no significant systematic bias'}")
print()
print("Correlation with epicentral distance")
print(f"  r = {r_dist:.3f},  p = {p_dist:.4f}  → "
      f"{'significant' if p_dist < 0.05 else 'not significant'}  "
      f"(slope = {slope_dist*1000:.2f} °/1000 km)")
print()
print("Correlation with station latitude")
print(f"  r = {r_lat:.3f},  p = {p_lat:.4f}  → "
      f"{'significant' if p_lat < 0.05 else 'not significant'}  "
      f"(slope = {slope_lat:.2f} °/°lat)")
print()
print("Interpretation:")
if p_val < 0.05 and abs(p_dist) > 0.05 and abs(p_lat) > 0.05:
    print("  Constant bias across all stations → consistent with a systematic")
    print("  sensor-level issue (e.g., all sensors rotated the same way) or")
    print("  a coherent regional anisotropy axis aligned with the path.")
elif p_dist < 0.05 or p_lat < 0.05:
    print("  Δbaz correlates with distance or latitude → consistent with")
    print("  crustal heterogeneity / anisotropy varying along the ray paths.")
else:
    print("  No statistically significant systematic trend. Offsets are likely")
    print("  station-specific sensor misorientation (random installation errors).")

# ── figure ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
cmap  = plt.get_cmap('RdBu_r')
vlim  = max(abs(delta_baz).max(), 30)
norm  = plt.Normalize(-vlim, vlim)

# (a) Map
ax = axes[0]
sc = ax.scatter(sta_lon, sta_lat, c=delta_baz, cmap=cmap, norm=norm,
                s=80, zorder=3, edgecolors='k', linewidths=0.4)
ax.scatter(EVT_LON, EVT_LAT, marker='*', s=250, color='gold',
           edgecolors='k', linewidths=0.5, zorder=5, label='Tohoku')
for i, sta in enumerate(stations):
    ax.annotate(sta.split('.')[1], (sta_lon[i], sta_lat[i]),
                fontsize=5, ha='center', va='bottom', xytext=(0, 4),
                textcoords='offset points')
plt.colorbar(sc, ax=ax, label='Δbaz (°)', shrink=0.8)
ax.set_xlabel('Longitude (°E)'); ax.set_ylabel('Latitude (°N)')
ax.set_title('Station map coloured by Δbaz\n(red = PCA > catalog, blue = PCA < catalog)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (b) Δbaz vs distance
ax = axes[1]
ax.scatter(distances, delta_baz, c=delta_baz, cmap=cmap, norm=norm,
           s=60, edgecolors='k', linewidths=0.4, zorder=3)
d_fit = np.array([distances.min(), distances.max()])
ax.plot(d_fit, slope_dist * d_fit + int_dist, 'k--', linewidth=1.2,
        label=f'r={r_dist:.2f}, p={p_dist:.3f}')
ax.axhline(0, color='gray', linewidth=0.8)
ax.axhline( 20, color='lightgray', linewidth=0.8, linestyle=':')
ax.axhline(-20, color='lightgray', linewidth=0.8, linestyle=':')
ax.set_xlabel('Epicentral distance (km)')
ax.set_ylabel('Δbaz (°)')
ax.set_title('Δbaz vs distance\n(path-length heterogeneity)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# (c) Δbaz vs latitude
ax = axes[2]
ax.scatter(sta_lat, delta_baz, c=delta_baz, cmap=cmap, norm=norm,
           s=60, edgecolors='k', linewidths=0.4, zorder=3)
lat_fit = np.array([sta_lat.min(), sta_lat.max()])
ax.plot(lat_fit, slope_lat * lat_fit + int_lat, 'k--', linewidth=1.2,
        label=f'r={r_lat:.2f}, p={p_lat:.3f}')
ax.axhline(0, color='gray', linewidth=0.8)
ax.axhline( 20, color='lightgray', linewidth=0.8, linestyle=':')
ax.axhline(-20, color='lightgray', linewidth=0.8, linestyle=':')
ax.set_xlabel('Station latitude (°N)')
ax.set_ylabel('Δbaz (°)')
ax.set_title('Δbaz vs latitude\n(spatial gradient)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

fig.suptitle('Systematicity of PCA back-azimuth offset  —  Δbaz = az_PCA − baz_catalog',
             fontsize=12)
plt.tight_layout()
plt.savefig('dbaz_systematicity.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
"""
GeoNet inventory orientation vs PCA Δbaz
-----------------------------------------
Query GeoNet FDSN at level='channel' to retrieve the recorded azimuth of the
HHN (and HH1) horizontal channels for each station at the time of the 2011
Tohoku event.

A perfectly-installed North sensor has azimuth = 0°.  Any deviation from 0°
is the sensor misorientation angle, which should predict the Δbaz we observe.

If:  inventory_misorientation  ≈  delta_baz  → sensor installation explains it
If:  no correlation             → path effects / anisotropy
"""
import numpy as np
import matplotlib.pyplot as plt
from obspy import UTCDateTime
from obspy.clients.fdsn import Client
from scipy import stats

EVT_TIME = UTCDateTime("2011-03-11T05:46:24")

client = Client("GEONET")
print("Querying GeoNet channel-level inventory …")
inv = client.get_stations(
    network="NZ", station="*", channel="HH?,BH?",
    starttime=EVT_TIME - 60, endtime=EVT_TIME + 3600,
    level="channel",
)
print("Done.\n")

# ── extract HHN (or HH1) azimuth per station ──────────────────────────────────
# GeoNet may label the N-component as HHN (azimuth≈0°) or HH1 (arbitrary az).
# We record the azimuth exactly as reported.
inv_az  = {}   # "NZ.STAZ" → recorded N-channel azimuth
inv_dip = {}

for net in inv:
    for sta in net:
        key = f"{net.code}.{sta.code}"
        for cha in sta:
            if cha.code in ("HHN", "HH1", "BHN", "BH1"):
                # prefer HH over BH; prefer N over 1
                if key not in inv_az or cha.code.startswith("HH"):
                    inv_az[key]  = cha.azimuth
                    inv_dip[key] = cha.dip

# ── match to our station list ─────────────────────────────────────────────────
inv_misori = np.full(nsta, np.nan)   # deviation of N-channel from geographic N
for i, sta_full in enumerate(stations):
    # stations stored as "NZ.RIZ" — strip the trailing Z for GeoNet name
    if sta_full in inv_az:
        inv_misori[i] = inv_az[sta_full]
    else:
        print(f"  {sta_full}: not found in inventory")

valid = ~np.isnan(inv_misori)

# ── compare inventory misorientation to PCA delta_baz ─────────────────────────
print(f"{'Station':<12} {'Inv N-az':>9} {'Δbaz (PCA)':>11} {'Diff':>7}")
print("-" * 45)
for i, sta_full in enumerate(stations):
    if valid[i]:
        diff = delta_baz[i] - inv_misori[i]
        diff = ((diff + 180) % 360) - 180
        print(f"{sta_full:<12} {inv_misori[i]:>9.1f}° {delta_baz[i]:>+10.1f}°  {diff:>+6.1f}°")
    else:
        print(f"{sta_full:<12} {'N/A':>9}  {delta_baz[i]:>+10.1f}°")

if valid.sum() >= 3:
    r, p = stats.pearsonr(inv_misori[valid], delta_baz[valid])
    slope, intercept, *_ = stats.linregress(inv_misori[valid], delta_baz[valid])
    print(f"\nCorrelation (inventory az vs PCA Δbaz): r = {r:.3f}, p = {p:.4f}")
    if p < 0.05:
        print("→ Significant: sensor misorientation explains the PCA offset.")
    else:
        print("→ Not significant: path effects / anisotropy more likely.")

    # ── scatter plot ──────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(inv_misori[valid], delta_baz[valid],
               s=65, color='#3498db', edgecolors='k', linewidths=0.5, zorder=3)
    x_fit = np.array([inv_misori[valid].min() - 5, inv_misori[valid].max() + 5])
    ax.plot(x_fit, slope * x_fit + intercept, 'k--', linewidth=1.2,
            label=f'r = {r:.2f}  p = {p:.3f}')
    ax.plot(x_fit, x_fit, color='gray', linewidth=0.8, linestyle=':',
            label='1 : 1  line')
    for i in np.where(valid)[0]:
        ax.annotate(stations[i].split('.')[1],
                    (inv_misori[i], delta_baz[i]),
                    fontsize=6, xytext=(3, 3), textcoords='offset points')
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.set_xlabel('Inventory N-channel azimuth  (°from N, clockwise)')
    ax.set_ylabel('PCA Δbaz  (az_PCA − baz_catalog, °)')
    ax.set_title('Sensor misorientation (inventory) vs PCA Δbaz\n'
                 'Alignment with 1:1 line → sensor error explains the offset')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('inventory_vs_pca_orientation.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Too few stations with inventory data to correlate.")
